[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fernfant/alexnet-explained/blob/main/build_your_own_alexnet.ipynb)


# Build Your Own AlexNet
### Episode 7 of *AlexNet in Six Episodes*: from watching to building

In 2012 Alex Krizhevsky trained AlexNet for 5–6 days on two gaming graphics cards in his bedroom.
Today you can rent a better GPU from Google **for free** and do the same kind of thing in minutes.

**What you'll build**

| Part | What you do | Time on a free GPU |
|---|---|---|
| 1 | Run the *real* AlexNet, show it your photos, look at the filters it learned | 2 min |
| 2 | Build **TinyAlexNet** from scratch (5 conv + 3 dense layers, ReLU, dropout, flips and crops) and train it on 50,000 photos | ~6 min |
| 3 | **Real-world project:** turn AlexNet into a *Pet Breed Detector* for 37 cat and dog breeds, then launch it as a web app you can open on your phone | ~5 min |

**By the end you will be able to**
* explain what a convolution layer, ReLU, dropout and data augmentation each do, and see them in real code
* train a neural network and read its loss and accuracy curves
* spot **overfitting** (great on training photos, worse on new ones)
* reuse a trained network for a new job with **transfer learning**, and ship it as an app

**How to use this notebook:** whenever you see **Predict first**, write down a guess before you run the cell. Being wrong and then finding out why is how you actually learn this. Each part ends with a short *Check yourself* quiz.

**Before you start (Google Colab):** `Runtime → Change runtime type → T4 GPU → Save`. Then run each cell with **Shift + Enter**.

In [ ]:
# ---- Setup: the tools we need ----
import time, io, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| training on:", device.upper())
if device == "cpu":
    print(" No GPU found. In Colab: Runtime → Change runtime type → T4 GPU. (CPU works, just slower.)")

---
## Part 1 · Meet the real AlexNet

PyTorch ships with AlexNet already trained on ImageNet's 1.2 million photos.
One small difference from 2012: this version fits on **one** GPU, so its first layer has 64 filters instead of 96.

In [ ]:
from torchvision.models import alexnet, AlexNet_Weights

weights = AlexNet_Weights.IMAGENET1K_V1          # the trained "knob settings"
model = alexnet(weights=weights).to(device).eval()
categories = weights.meta["categories"]          # the 1,000 ImageNet labels

print(model)
print(f"\nKnobs (parameters): {sum(p.numel() for p in model.parameters()):,}")
print(f"Categories: {len(categories)}, e.g. {categories[281]!r}, {categories[779]!r}, {categories[949]!r}")

**Predict first:** the photo goes in as 3 × 224 × 224. Will the grid be bigger or smaller after the first layer? Will there be more channels or fewer?

In [ ]:
# Follow one photo through the network: the grid gets smaller but deeper (just like Episode 4)
x = torch.zeros(1, 3, 224, 224, device=device)
print(f"{'Input':12s} {tuple(x.shape[1:])}")
with torch.no_grad():
    for layer in model.features:
        x = layer(x)
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            print(f"{layer.__class__.__name__:12s} {tuple(x.shape[1:])}")

In [ ]:
# ---- Helpers: load a photo and ask AlexNet for its top 5 guesses ----
preprocess = weights.transforms()   # resize to 224x224 and scale the numbers like in training

def load_image(src):
    # Load a photo from a web address or a file name.
    if str(src).startswith("http"):
        req = urllib.request.Request(src, headers={"User-Agent": "Mozilla/5.0"})
        return Image.open(io.BytesIO(urllib.request.urlopen(req).read())).convert("RGB")
    return Image.open(src).convert("RGB")

def top5(img):
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model(x).softmax(dim=1)[0]
    p, idx = probs.topk(5)
    plt.figure(figsize=(9, 3.2))
    plt.subplot(1, 2, 1); plt.imshow(img); plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.barh([categories[i] for i in idx.tolist()][::-1], (p.cpu().numpy() * 100)[::-1], color="#f2a541")
    plt.xlabel("confidence %"); plt.xlim(0, 100); plt.tight_layout(); plt.show()

top5(load_image("https://github.com/pytorch/hub/raw/master/images/dog.jpg"))

In [ ]:
# ---- Try your own photo ----
# Set this to True, run the cell, and pick a photo (your pet, your shoes, a football, anything).
USE_MY_OWN_PHOTO = False

if USE_MY_OWN_PHOTO:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        top5(load_image(name))

### Be the filter
Episode 3 showed a 3×3 filter sliding over a picture. Here is the same thing in one line of PyTorch, `F.conv2d`, on a real photo.
Change the numbers in `edge` and re-run. Can you make it find **horizontal** edges instead?

In [ ]:
# ---- Be the filter: slide your own 3x3 filter over a photo ----
photo = load_image("https://github.com/pytorch/hub/raw/master/images/dog.jpg").resize((320, 240))
gray = T.functional.to_tensor(photo.convert("L")).unsqueeze(0)      # 1 photo x 1 channel x 240 x 320

edge = torch.tensor([[-1., 0., 1.],
                     [-1., 0., 1.],
                     [-1., 0., 1.]])                                  # dark on the left, bright on the right
out = F.conv2d(gray, edge.view(1, 1, 3, 3), padding=1)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(gray[0, 0], cmap="gray"); ax[0].set_title("photo (brightness)")
ax[1].imshow(out[0, 0], cmap="coolwarm", vmin=-out.abs().max(), vmax=out.abs().max()); ax[1].set_title("filter output (red +, blue -)")
ax[2].imshow(out[0, 0].relu(), cmap="inferno"); ax[2].set_title("after ReLU (negative -> 0)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Look inside: the 64 filters AlexNet taught itself ----
# Nobody drew these. They came out of training. Look for stripes (edge detectors) and colour blobs.
w = model.features[0].weight.detach().cpu()                    # shape: 64 filters x 3 colours x 11 x 11
grid = torchvision.utils.make_grid(w, nrow=16, padding=1, normalize=True, scale_each=True)
plt.figure(figsize=(12, 3.4)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
plt.title("AlexNet's first-layer filters (each one is 11x11 pixels)"); plt.show()

### Check yourself: Part 1
Try to answer in your head before you click.

**1. AlexNet's grid went from 224×224 to 6×6, but from 3 channels to 256. Why is that a good trade?**
<details><summary>Show answer</summary>

Each channel is a pattern detector. Deeper layers care less about *where* something is and more about *what* it is, so they keep fewer positions and many more patterns.
</details>

**2. Nobody designed the filters you just looked at. Where did they come from?**
<details><summary>Show answer</summary>

They started as random numbers. Training nudged them, photo after photo, until they became edge and colour detectors, because those help reduce mistakes.
</details>

**3. Your filter output had negative numbers. What does ReLU do to them, and why is that useful?**
<details><summary>Show answer</summary>

ReLU turns them into 0. A filter then only 'speaks up' where its pattern is, and stays silent everywhere else.
</details>

---
## Part 2 · Build TinyAlexNet from scratch

The real AlexNet needed 1.2 million big photos and days of training. We'll keep the same **recipe** and shrink the **ingredients**:

* **Data:** CIFAR-10, 50,000 tiny 32×32 photos in 10 classes (made at the University of Toronto, partly by Alex himself!)
* **Network:** 5 convolution layers + 3 dense layers, with ReLU and dropout, the same shape as AlexNet
* **Tricks:** random crops and flips, exactly like 2012

In [ ]:
# ---- The data ----
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),      # trick 3: shift the photo around a bit...
    T.RandomHorizontalFlip(),         # ...and mirror it half the time
    T.ToTensor(),                     # pixels -> numbers between 0 and 1
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_set = torchvision.datasets.CIFAR10("data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
classes = train_set.classes
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f"{len(train_set):,} training photos, {len(test_set):,} test photos, classes: {classes}")

def show_grid(images, titles, mean, std, colors=None, n=16, cols=8):
    # Undo the normalising so the photos look normal again, then plot them.
    m, s = torch.tensor(mean).view(3, 1, 1), torch.tensor(std).view(3, 1, 1)
    rows = (min(n, len(images)) + cols - 1) // cols
    plt.figure(figsize=(cols * 1.6, rows * 1.9))
    for i in range(min(n, len(images))):
        plt.subplot(rows, cols, i + 1)
        plt.imshow((images[i].cpu() * s + m).clamp(0, 1).permute(1, 2, 0))
        plt.title(titles[i], fontsize=8, color=(colors[i] if colors else "black")); plt.axis("off")
    plt.tight_layout(); plt.show()

imgs, labels = next(iter(train_loader))
show_grid(imgs, [classes[l] for l in labels], CIFAR_MEAN, CIFAR_STD)

**Predict first:** TinyAlexNet has the same 8 layers as AlexNet, but takes 32×32 photos instead of 224×224. Will it have more or fewer knobs than the real one's 61 million? Roughly how many?

In [ ]:
# ---- The network: same shape as AlexNet, sized for 32x32 photos ----
class TinyAlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # 5 convolution layers: find patterns (edges -> shapes -> parts)
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),    nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(64, 192, kernel_size=3, padding=1),  nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 8 -> 4
        )
        # 3 dense layers: weigh up the evidence and decide
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 4 * 4, 1024), nn.ReLU(inplace=True),   # trick 2: dropout
            nn.Dropout(0.5), nn.Linear(1024, 1024),        nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes),                  # one score per class
        )
        # Start the conv knobs at a sensible random size ("He initialisation") so training gets going fast
        for m in self.features:
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

tiny = TinyAlexNet()
x = torch.zeros(1, 3, 32, 32)
for layer in tiny.features:
    x = layer(x)
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
        print(f"{layer.__class__.__name__:10s} {tuple(x.shape[1:])}")
print(f"\nTinyAlexNet knobs: {sum(p.numel() for p in tiny.parameters()):,}  (the real one has ~61 million)")
tiny = tiny.to(device)

In [ ]:
# ---- Training: guess, measure the mistake, nudge every knob (Episode 5) ----
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train(training)                           # dropout only switches on while training
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            scores = model(x)                       # 1. guess
            loss = F.cross_entropy(scores, y)       # 2. how wrong was it?
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()                     # 3. backpropagation: which knobs to blame?
                optimizer.step()                    # 4. nudge every knob a tiny bit
                if scheduler is not None:
                    scheduler.step()
            total_loss += loss.item() * len(y)
            correct += (scores.argmax(dim=1) == y).sum().item()
            seen += len(y)
    return total_loss / seen, correct / seen

def fit(model, train_loader, test_loader, epochs, optimizer, scheduler=None):
    history = {"loss": [], "train_acc": [], "test_acc": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        loss, train_acc = run_epoch(model, train_loader, optimizer, scheduler)
        _, test_acc = run_epoch(model, test_loader)
        for k, v in zip(history, (loss, train_acc, test_acc)):
            history[k].append(v)
        print(f"pass {epoch:2d}/{epochs} | mistakes score {loss:.3f} | "
              f"train {train_acc:6.1%} | test {test_acc:6.1%} | {time.time() - start:4.0f}s")
    return history

def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].plot(h["loss"], color="#ef6f6c", lw=2); ax[0].set_title("Mistakes score (loss): should fall")
    ax[1].plot(h["train_acc"], label="train", lw=2, color="#a99bdf")
    ax[1].plot(h["test_acc"], label="test (never seen)", lw=2, color="#f2a541")
    ax[1].set_title("Accuracy: should rise"); ax[1].legend()
    for a in ax: a.set_xlabel("pass through the data (epoch)"); a.grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()

**Predict first:** with 10 classes, what accuracy would random guessing get? Write down where you think the *test* accuracy will be after pass 1 and after the last pass.

While it trains, watch three numbers: the **mistakes score** (loss) should fall, and **train** and **test** accuracy should rise. If train keeps climbing but test stalls, the network has started memorising.

In [ ]:
# ---- Go! ----
EPOCHS = 15 if device == "cuda" else 3
# AlexNet used plain SGD. We use AdamW (2017), a smarter knob-nudger that adapts the step size for every knob.
optimizer = torch.optim.AdamW(tiny.parameters(), lr=1e-3, weight_decay=5e-4)
# Start with small nudges, speed up, then slow down again to settle in
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, epochs=EPOCHS,
                                                steps_per_epoch=len(train_loader))
# On a T4 GPU expect about 20-30 seconds per pass, ending at roughly 80% on photos it has never seen.
tiny_history = fit(tiny, train_loader, test_loader, EPOCHS, optimizer, scheduler)
plot_history(tiny_history, "TinyAlexNet on CIFAR-10")

In [ ]:
# ---- How did it do? Green = right, red = wrong ----
tiny.eval()
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    preds = tiny(imgs.to(device)).argmax(1).cpu()
show_grid(imgs, [f"{classes[p]}\n(true: {classes[l]})" for p, l in zip(preds, labels)],
          CIFAR_MEAN, CIFAR_STD, colors=["green" if p == l else "red" for p, l in zip(preds, labels)], n=24)

# Which classes are hardest?
correct, total = torch.zeros(10), torch.zeros(10)
with torch.no_grad():
    for x, y in test_loader:
        p = tiny(x.to(device)).argmax(1).cpu()
        for c in range(10):
            total[c] += (y == c).sum(); correct[c] += ((y == c) & (p == c)).sum()
for c in (correct / total).argsort():
    print(f"{classes[c]:>10s}  {correct[c] / total[c]:6.1%}")

### Check yourself: Part 2
Try to answer in your head before you click.

**1. Random guessing on CIFAR-10 scores 10%. Why?**
<details><summary>Show answer</summary>

There are 10 classes, so a random guess is right 1 time in 10.
</details>

**2. Why is the **test** accuracy the one that matters, not the train accuracy?**
<details><summary>Show answer</summary>

The test photos were never used for training. They show how the network does on new photos, which is the whole point. High train accuracy alone could just mean it memorised.
</details>

**3. What are `loss.backward()` and `optimizer.step()` doing?**
<details><summary>Show answer</summary>

`backward()` is backpropagation: it works out how much each knob is to blame for the mistake. `step()` nudges every knob a little in the direction that reduces the mistake.
</details>

**4. Which classes were hardest? Why might cats vs dogs be harder than ships vs frogs?**
<details><summary>Show answer</summary>

Cats and dogs share fur, ears, eyes and poses, especially at 32×32 pixels. Ships and frogs look nothing alike, and usually have different backgrounds (sea vs grass).
</details>

---
## Part 3 · Real-world project: Pet Breed Detector
Training from scratch needs **lots** of photos. Real engineers usually don't start from zero. They take a network
that already knows how to see (AlexNet's edge, shape and part detectors) freeze those layers, and retrain only the **deciding layers** for a new job, starting with a fresh last layer.
This is called **transfer learning**.

**Dataset:** Oxford-IIIT Pets, about 7,400 photos of 37 cat and dog breeds, from the University of Oxford.

In [ ]:
# ---- The data: photos of 37 breeds ----
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

pet_train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),    # random crops
    T.RandomHorizontalFlip(),                      # random flips
    T.ColorJitter(0.2, 0.2, 0.2),                  # small colour shifts (AlexNet did this too)
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
pet_test_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

pet_train = torchvision.datasets.OxfordIIITPet("data", split="trainval", download=True, transform=pet_train_tf)
pet_test  = torchvision.datasets.OxfordIIITPet("data", split="test",     download=True, transform=pet_test_tf)
pet_classes = pet_train.classes
pet_train_loader = DataLoader(pet_train, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
pet_test_loader  = DataLoader(pet_test,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f"{len(pet_train):,} training photos, {len(pet_test):,} test photos, {len(pet_classes)} breeds")

imgs, labels = next(iter(pet_train_loader))
show_grid(imgs, [pet_classes[l] for l in labels], IMAGENET_MEAN, IMAGENET_STD)

In [ ]:
# ---- Transfer learning: keep AlexNet's eyes, give it a new brain for breeds ----
pet_model = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)

for p in pet_model.features.parameters():
    p.requires_grad = False                         # freeze the 5 conv layers: they already know how to see

pet_model.classifier[6] = nn.Linear(4096, len(pet_classes))   # swap 1,000 ImageNet answers for 37 breeds
pet_model = pet_model.to(device)

trainable = sum(p.numel() for p in pet_model.parameters() if p.requires_grad)
print(f"Knobs we will train: {trainable:,} of {sum(p.numel() for p in pet_model.parameters()):,}")

In [ ]:
PET_EPOCHS = 6 if device == "cuda" else 2
pet_opt = torch.optim.Adam([p for p in pet_model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)
pet_history = fit(pet_model, pet_train_loader, pet_test_loader, PET_EPOCHS, pet_opt)
plot_history(pet_history, "Pet Breed Detector (AlexNet + transfer learning)")
torch.save(pet_model.state_dict(), "pet_breed_alexnet.pt")
print("Saved the trained model to pet_breed_alexnet.pt")

In [ ]:
# ---- Test it on photos it has never seen ----
def predict_breed(img, k=3):
    pet_model.eval()
    with torch.no_grad():
        probs = pet_model(pet_test_tf(img).unsqueeze(0).to(device)).softmax(1)[0].cpu()
    p, idx = probs.topk(k)
    return [(pet_classes[i], float(pp)) for pp, i in zip(p, idx)]

pet_model.eval()
imgs, labels = next(iter(pet_test_loader))
with torch.no_grad():
    preds = pet_model(imgs.to(device)).argmax(1).cpu()
show_grid(imgs, [f"{pet_classes[p]}\n(true: {pet_classes[l]})" for p, l in zip(preds, labels)],
          IMAGENET_MEAN, IMAGENET_STD, colors=["green" if p == l else "red" for p, l in zip(preds, labels)],
          n=16, cols=4)

# Your own pet? Set to True and upload a photo.
USE_MY_OWN_PET = False
if USE_MY_OWN_PET:
    from google.colab import files
    for name in files.upload():
        img = load_image(name)
        plt.imshow(img); plt.axis("off"); plt.show()
        for breed, p in predict_breed(img):
            print(f"{breed:>28s}  {p:6.1%}")

### Check yourself: Part 3
Try to answer in your head before you click.

**1. Why can the pet detector reach good accuracy in a few minutes from only about 3,700 training photos?**
<details><summary>Show answer</summary>

Its 5 convolution layers already learned to see from 1.2 million ImageNet photos. We only train the deciding layers, which is a much smaller job.
</details>

**2. What does `requires_grad = False` do?**
<details><summary>Show answer</summary>

It freezes those knobs: backpropagation skips them, so training doesn't change them.
</details>

**3. Why replace `classifier[6]`?**
<details><summary>Show answer</summary>

It was the last layer, with 1,000 outputs for ImageNet's categories. We need 37 outputs, one per breed.
</details>

### Launch it as a real app
The next cell turns your model into a web app. Colab prints a **public link** (it works for 72 hours) that you can open
on your phone, take a photo of a cat or dog, and see what your AlexNet thinks.

In [ ]:
%pip install -q gradio
import gradio as gr

def app_predict(img):
    if img is None:
        return {}
    return dict(predict_breed(img.convert("RGB"), k=5))

gr.Interface(
    fn=app_predict,
    inputs=gr.Image(type="pil", label="Photo of a cat or dog"),
    outputs=gr.Label(num_top_classes=5, label="Breed"),
    title="Pet Breed Detector",
    description="Built on AlexNet (2012) with transfer learning. Trained in minutes on one free GPU.",
).launch(share=True)

---
## Level up: challenges

1. **Beat 2012 on CIFAR-10.** Change `EPOCHS` to 30. Does test accuracy keep climbing, or does it level off?
2. **Go old-school.** Replace AdamW with AlexNet's original optimiser: `torch.optim.SGD(tiny.parameters(), lr=0.02, momentum=0.9, weight_decay=5e-4)` (and `max_lr=0.02`). Is it slower to get going?
3. **Remove a trick.** Delete the `nn.Dropout` lines, or the random crop and flip, and retrain. How much worse does the *test* score get compared with the *train* score? That gap is called **overfitting**.
4. **Unfreeze AlexNet.** In Part 3, skip the freezing loop and use `lr=1e-4`. Does fine-tuning *all* the knobs help?
5. **Use a modern network.** Swap `alexnet` for `torchvision.models.resnet18` (2015). Replace `model.fc` instead of `classifier[6]`. How much better is it?
6. **Your own dataset.** Make folders like `my_data/train/football_boots/`, `my_data/train/trainers/` with 30+ photos each, and load them with
   `torchvision.datasets.ImageFolder("my_data/train", transform=pet_train_tf)`. Everything else in Part 3 stays the same.

**The big idea:** the recipe from 2012 (lots of labelled data, layers of learned filters, ReLU, dropout, data augmentation, and a GPU)
is the same recipe behind almost every AI system you use today.

---
## Recap: what you just did

| Idea from 2012 | Where it was in your code |
|---|---|
| Convolution filters find patterns | `nn.Conv2d` layers, and your own `F.conv2d` filter |
| ReLU: negative → 0 | `nn.ReLU` after every layer |
| Dropout fights memorising | `nn.Dropout(0.5)` in the dense layers |
| Data augmentation: free extra photos | `RandomCrop`, `RandomHorizontalFlip`, `ColorJitter` |
| Learning from mistakes | `loss.backward()` then `optimizer.step()` |
| Train on a GPU | `.to(device)` with `device = "cuda"` |

**Share it:** send the Gradio link to a friend, or save your notebook to Drive and show someone your accuracy curves.

**Want to go further?** Try the challenges above, then look up ResNet (2015), which made networks 100+ layers deep, and Vision Transformers (2020), which many image models use today.